> patch prediction will be converted to whole image prediction

In [ ]:
#| default_exp model_eval.frm_patch_model_whole_model

: 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from fastcore.script import *
from pathlib import Path
import sys

In [ ]:
#| export
custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
sys.path.append(str(custom_lib_path))
CURRENT_DIR = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection')
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
NORMAL_FRONT_PIN = Path(r'/home/ai_sintercra/homes/hasan/easy_front/easy-front-pin-detection')
import sys
sys.path.append(str(CURRENT_DIR))
sys.path.append(str(CV_TOOLS))
sys.path.append(str(NORMAL_FRONT_PIN))

In [ ]:
#| export
import tensorflow as tf

2024-09-05 14:30:59.642447: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-09-05 14:30:59.791863: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/ai_warstein/data/share/oracle_instantclient_12_2:/opt/itsdm/packages/mosquitto-1.6.2/lib:/opt/itsdm/packages/libwebsocket-2.4.2/lib:/opt/lsf_10.1.0/10.1/linux3.10-glibc2.17-x86_64/lib
2024-09-05 14:30:59.791894: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-09-05 14:31:01.567437: W tensor

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
from private_easy_pin_detection.imports import *
from private_easy_pin_detection.mask_generation_patch_image125 import *
from private_easy_pin_detection.model_eval.two_image_model import *

In [ ]:
from cv_tools.imports import *

In [ ]:
#| export
def reconstruct_from_patches(
        patches:tf.Tensor, 
        original_shape:tuple[int, int], 
        patch_size:int=128
        ):
    ' Reconstruction of the image from patches'

    # Assuming patches are square and the batch dimension is flat
    num_patches_per_row = tf.math.ceil(tf.cast(original_shape[1], float) / patch_size)
    num_patches_per_col = tf.math.ceil(tf.cast(original_shape[2], float) / patch_size)
    
    num_patches_per_row = tf.cast(num_patches_per_row, tf.int32)
    num_patches_per_col = tf.cast(num_patches_per_col, tf.int32)
    patch_size = tf.cast(patch_size, tf.int32)
    # Reshape to grid
    patches = tf.reshape(patches, (
        -1, 
        num_patches_per_row, 
        num_patches_per_col, 
        patch_size, 
        patch_size, 
        original_shape[-1]))
    patches = tf.transpose(patches, [0, 1, 3, 2, 4, 5])
    patches = tf.reshape(
        patches, 
        (
            -1, 
            tf.cast(num_patches_per_row * patch_size, tf.int32),
            tf.cast(num_patches_per_col * patch_size, tf.int32),
            tf.cast(original_shape[-1], tf.int32)
        ))
    
    # Crop to original dimensions
    return patches[:, :original_shape[1], :original_shape[2], :]


In [ ]:
#| export
def frm_patch_mdl_to_one_img_mdl( 
    patch_config:dict,

    save_:False
    
    
): 
    config=patch_config
    inputs_ = tf.keras.layers.Input(
    shape= tuple(config['model']['input_shape'])
    )
    patches = split_image_to_patches(inputs_, config['dataset']['patch_size'])
    patch_model_fn = Path(config['model']['patch_model_save_path'], config['model']['patch_model_name'])
    old_model = tf.keras.models.load_model(
                                       f'{patch_model_fn}',
                                        compile=False
                                        )
    
    patch_pred = old_model(patches)
    reconstructed_image=reconstruct_from_patches(
        patch_pred, 
        original_shape=tuple(config['model']['batch_input_shape']),
        patch_size=config['dataset']['patch_size'])
    prd = reconstructed_image > config['model']['threshold']
    model_ = tf.keras.models.Model(
                                    inputs=[inputs_],
                                    outputs=[prd]
                                    )
    one_image_mdl_full_fn=Path(config['model']['one_image_model_save_path'], config['model']['patch_model_name'])
    if save_:
        Path(config['model']['one_image_model_save_path']).mkdir(exist_ok=True, parents=True)
        tf.keras.models.save_model(model_,f'{one_image_mdl_full_fn}')
    return model_   

In [ ]:
def frm_patch_mdl_to_two_img_mdl(
                                patch_config:dict,
                                save_:False):
    _ = frm_patch_mdl_to_one_img_mdl(patch_config, save_=save_)
    

In [ ]:
Path.cwd()

Path('/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/nbs')

In [ ]:
patch_model_config_fn=Path('/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_test_patch_to_normal_model.yml')
patch_config=read_config(patch_model_config_fn)

In [ ]:
patch_config

{'dataset': {'name': 'SingleChannelPatch256Dataset',
  'image_height': 1152,
  'image_width': 1632,
  'min': 0,
  'image_channels': 1,
  'min_contour_area': 1,
  'patch_size': 256},
 'model': {'architecture': 'UNetSmall',
  'input_shape': [1152, 1632, 1],
  'num_classes': 1,
  'threshold': 0.5,
  'patch_model_save_path': '/home/ai_sintercra.work/data/projects/easy_pin_detection/models/segmentation_test/patch_256_20240903res_att_Unet',
  'patch_model_name': 'time_07_19_13_val_frGrnd__0.9638_epoch_189.h5',
  'one_image_model_save_path': '/home/ai_sintercra.work/data/projects/easy_pin_detection/PatchModel/one_image_model',
  'two_image_model_save_path': '/home/ai_sintercra.work/data/projects/easy_pin_detection/PatchModel/two_image_model'}}

In [ ]:
m_c = patch_config['model']
d_c = patch_config['dataset']
fn =Path(m_c['patch_model_save_path'], m_c['patch_model_name'])

In [ ]:
mdl=tf.keras.models.load_model(f'{fn}', compile=False)

In [ ]:
 inputs_ = tf.keras.layers.Input(
    shape= tuple(m_c['input_shape'])
    )

In [ ]:
patches_ = split_image_to_patches(
                                 inputs_,
                                  d_c['patch_size'])
patch_pred = mdl(patches_)
reconstructed_image=reconstruct_from_patches(
        patch_pred, 
        original_shape=tuple(m_c['input_shape']),
        patch_size=d

In [ ]:
patches_ = split_image_to_patches(
                                 inputs_,
                                  d_c['patch_size'])
patch_pred = mdl(patches_)
reconstructed_image=reconstruct_from_patches(
        patch_pred, 
        original_shape=m_c['batch_input_shape'],
        patch_size=d_c['patch_size'],
)

In [ ]:
m_c['input_sah']

In [ ]:
patches_.shape

TensorShape([None, 256, 256, 1])

In [ ]:
reconstructed_image.shape

TensorShape([None, 1152, 1632, 1])

In [ ]:
one_image_mdl = frm_patch_mdl_to_one_img_mdl(
                                            patch_config=patch_config, 
                                            save_=True)

In [ ]:
one_image_mdl.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_13 (InputLayer)       [(None, 1152, 1632, 1)]   0         
                                                                 
 tf.compat.v1.pad_18 (TFOpLa  (None, 1280, 1792, 1)    0         
 mbda)                                                           
                                                                 
 tf.reshape_44 (TFOpLambda)  (None, 5, 256, 7, 256, 1  0         
                             )                                   
                                                                 
 tf.compat.v1.transpose_22 (  (None, 5, 7, 256, 256, 1  0        
 TFOpLambda)                 )                                   
                                                                 
 tf.reshape_45 (TFOpLambda)  (None, 256, 256, 1)       0         
                                                           

In [ ]:
current_model_path = Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/models/segmentation_test/patch_256_20240903res_att_Unet/time_07_19_13_val_frGrnd__0.9638_epoch_189.h5')
patch_model = tf.keras.models.load_model(current_model_path, compile=False)

In [ ]:
patch_model_path=Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/models/segmentation_test/patch_256_20240903res_att_Unet/time_07_19_13_val_frGrnd__0.9571_epoch_121.h5')

In [ ]:
#patch_model.summary()

In [ ]:
from fastcore.script import *

In [ ]:
##| export
#@call_parse
#def main_():
    #patch_config=read_config(PathParam('config_path', str))
    #print('config reading is done')
    #print('Started creating model ..')
    #frm_patch_mdl_to_one_img_mdl(
        #Param('save', action=store_true),
        #patch_config=patch_config, 
    #)

In [ ]:
#| export
def parse_args_():
    from argparse import ArgumentParser
    parser = ArgumentParser()
    parser.add_argument('--config_path', type=str)
    parser.add_argument('--save', action='store_true')
    return parser.parse_args()
    

In [ ]:
#| export
def main_():
    args = parse_args_()
    patch_config=read_config(Path(args.config_path))
    frm_patch_mdl_to_one_img_mdl(
        patch_config=patch_config, 
        save_=args.save
    )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('32_model_evaluation.frm_patch_model_whole_model.ipynb')